In [6]:
# ==========================================
# PROYECTO FINAL IA GENERATIVA - CONFIGURACIÓN
# ==========================================

uuid_alumno = "917343c0-dc7f-4300-85cd-651670e304c2" 

import os
from dotenv import load_dotenv

# Cargar el archivo .env
load_dotenv()

# Leer la variable exacta que tienes en tu archivo
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    print("Error: No se encuentra 'GEMINI_API_KEY'. Verifica el nombre en el archivo .env")
else:
    # Asignarla a la variable que usará LangChain (GOOGLE_API_KEY)
    os.environ["GOOGLE_API_KEY"] = api_key
    print("¡ÉXITO! API Key cargada correctamente.")

¡ÉXITO! API Key cargada correctamente.


In [7]:
# ==========================================
# PASO 2 Y 3: CARGA Y CREACIÓN DE CHROMA DB
# ==========================================

import json
import os
import time
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

carpeta_datos = "data"
archivos_json = ['league_of_legends_champions.json', 'league_of_legends_items.json', 'league_of_legends_macro.json']
documentos_totales = []

for archivo in archivos_json:
    ruta_completa = os.path.join(carpeta_datos, archivo)
    try:
        with open(ruta_completa, 'r', encoding='utf-8') as f:
            datos = json.load(f)
            for item in datos:
                contenido = json.dumps(item, ensure_ascii=False)
                doc = Document(page_content=contenido, metadata={"source": archivo})
                documentos_totales.append(doc)
        print(f"✅ Cargado: {ruta_completa}")
    except FileNotFoundError:
        print(f"❌ Error: No se encuentra el archivo en {ruta_completa}")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documentos_totales)

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

vector_db = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)

chunk_length = 5
for i in range(0, len(chunks), chunk_length):
    lote = chunks[i:i + chunk_length]
    vector_db.add_documents(lote)
    time.sleep(6)
    print(f"Procesado lote {i // chunk_length + 1}")

print(f"\n🚀 Base de conocimiento creada con {len(chunks)} fragmentos.")

test_query = "Cuál es la mejor build para un campeón?"
resultados = vector_db.similarity_search(test_query, k=2)

print(f"\n🔍 Test de recuperación para: '{test_query}'")
for i, res in enumerate(resultados):
    print(f"Resultado {i+1}: {res.page_content[:100]}...")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Cargado: data\league_of_legends_champions.json
✅ Cargado: data\league_of_legends_items.json
✅ Cargado: data\league_of_legends_macro.json
Procesado lote 1
Procesado lote 2
Procesado lote 3
Procesado lote 4
Procesado lote 5
Procesado lote 6
Procesado lote 7
Procesado lote 8
Procesado lote 9
Procesado lote 10
Procesado lote 11
Procesado lote 12
Procesado lote 13
Procesado lote 14
Procesado lote 15
Procesado lote 16
Procesado lote 17
Procesado lote 18
Procesado lote 19
Procesado lote 20
Procesado lote 21
Procesado lote 22
Procesado lote 23
Procesado lote 24
Procesado lote 25
Procesado lote 26
Procesado lote 27
Procesado lote 28
Procesado lote 29
Procesado lote 30
Procesado lote 31
Procesado lote 32
Procesado lote 33
Procesado lote 34
Procesado lote 35
Procesado lote 36
Procesado lote 37
Procesado lote 38
Procesado lote 39
Procesado lote 40
Procesado lote 41
Procesado lote 42
Procesado lote 43
Procesado lote 44
Procesado lote 45
Procesado lote 46
Procesado lote 47
Procesado lote 48
Proces

In [ ]:
from typing import TypedDict, List
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

class AgentState(TypedDict):
    question: str
    context: str
    answer: str
    history: List[BaseMessage]
    source: str

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vector_db = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)

retriever = vector_db.as_retriever(search_kwargs={"k": 3})

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

system_prompt_rag = (
    "Eres un Analista Experto en League of Legends de Nivel Competitivo y un asistente RAG especializado.\n\n"
    "Tu misión principal es analizar los datos proporcionados en el contexto (JSON/documentos) y usar tu capacidad de razonamiento "
    "para ofrecer recomendaciones estratégicas personalizadas al usuario.\n\n"
    "Reglas de funcionamiento:\n"
    "1. Base de Verdad: Utiliza la información técnica de los campeones, objetos y mecánicas presentes en el contexto como cimiento "
    "de tu respuesta. No inventes estadísticas que no existan.\n"
    "2. Razonamiento Estratégico: Si el usuario plantea un escenario (ej. 'mi equipo es full AP'), integra los datos de tu base "
    "con conceptos de estrategia (meta, sinergias, counters) para dar una respuesta completa. No te limites a citar el texto; "
    "interpreta los datos para dar una solución.\n"
    "3. Integridad: Si la pregunta requiere datos que NO están en tu base vectorial (ej. 'qué tiempo hace hoy' o 'dime un chiste'), "
    "responde textualmente: 'No dispongo de esa información en mi base de conocimiento local.'\n\n"
    "Formato de Respuesta:\n"
    "- Usa negritas para destacar conceptos clave.\n"
    "- Estructura tu razonamiento en puntos claros.\n"
    "- Limita tu recomendación a un máximo de 2 campeones y, para cada campeón, aporta únicamente dos viñetas breves con la razón táctica básica.\n"
    "- Sé directo, profesional y al grano.\n\n"
    "Contexto adjunto:\n"
    "{context}"
)

system_prompt_fallback = (
    "Eres un Asistente de Análisis de League of Legends. "
    "Tu función es informar al usuario de que la información solicitada no se encuentra en la base de datos local disponible. "
    "No intentes responder con conocimiento general ni inventar datos. "
    "Tu respuesta debe ser siempre: 'Lo siento, no dispongo de esa información técnica específica en mi base de datos. Por favor, intenta reformular tu pregunta sobre campeones, objetos o estrategias incluidas en mis registros.'"
)
prompt_rag = ChatPromptTemplate.from_messages([
    ("system", system_prompt_rag),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

prompt_fallback = ChatPromptTemplate.from_messages([
    ("system", system_prompt_fallback),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

def retrieve_node(state: AgentState):
    question = state["question"]
    try:
        docs = retriever.invoke(question)
        context = "\n\n".join(doc.page_content for doc in docs)
    except Exception:
        context = ""
    return {"context": context}

def generate_rag_node(state: AgentState):
    question = state["question"]
    context = state["context"]
    history = state.get("history", [])
    chain = prompt_rag | llm
    response = chain.invoke({"context": context, "history": history, "input": question})
    updated_history = history + [HumanMessage(content=question), AIMessage(content=response.content)]
    return {"answer": response.content, "history": updated_history, "source": "Base de Datos Local (RAG - JSON)"}

def generate_fallback_node(state: AgentState):
    question = state["question"]
    history = state.get("history", [])
    chain = prompt_fallback | llm
    response = chain.invoke({"history": history, "input": question})
    updated_history = history + [HumanMessage(content=question), AIMessage(content=response.content)]
    return {"answer": response.content, "history": updated_history, "source": "Conocimiento General (Fallback)"}

def checker_router(state: AgentState):
    answer_str = str(state["answer"]).lower()
    if "no dispongo de esa información" in answer_str:
        return "fallback"
    return "end"

workflow = StateGraph(AgentState)

workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate_rag", generate_rag_node)
workflow.add_node("generate_fallback", generate_fallback_node)

workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "generate_rag")

workflow.add_conditional_edges(
    "generate_rag",
    checker_router,
    {
        "fallback": "generate_fallback",
        "end": END
    }
)

workflow.add_edge("generate_fallback", END)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [11]:
from IPython.display import display, Markdown

config = {"configurable": {"thread_id": "chat_interactivo_lol"}}

print("=== CONVERSACIÓN CON EL ASISTENTE EXPERTO EN LEAGUE OF LEGENDS ===")
print("Escribe 'salir' para cerrar el canal táctico.\n")

while True:
    user_input = input("Introduce tu consulta: ")
    if user_input.lower() in ["salir", "exit", "quit"]:
        print("\nCanal cerrado. ¡Buena suerte en la Grieta, Invocador!🧙‍♂️")
        break
        
    if not user_input.strip():
        continue
        
    try:
        response = app.invoke({"question": user_input}, config=config)
        raw_answer = response.get('answer', '')
        origen_datos = response.get('source', 'Desconocido')
        
        if isinstance(raw_answer, list) and len(raw_answer) > 0:
            clean_answer = raw_answer[0].get('text', str(raw_answer))
        elif isinstance(raw_answer, dict):
            clean_answer = raw_answer.get('text', str(raw_answer))
        else:
            clean_answer = str(raw_answer)
            
        print(f"\n🔹 Invocador: {user_input}")
        print("\n🔸 Asistente Experto:")
        display(Markdown(clean_answer))
        
        print(f"\n[🔧 ORIGEN DE LA RESPUESTA: {origen_datos}]")
        print("-" * 60)
    except Exception as e:
        print(f"\n❌ Error en el sistema: {e}\n")

=== CONVERSACIÓN CON EL ASISTENTE EXPERTO EN LEAGUE OF LEGENDS ===
Escribe 'salir' para cerrar el canal táctico.


🔹 Invocador: Con Aatrox, ¿qué habilidad debo priorizar al nivel 1 para asegurar un tradeo favorable y cuál es mi mayor power spike en los primeros niveles?

🔸 Asistente Experto:


Para asegurar un tradeo favorable con **Aatrox** en el nivel 1, tu elección estratégica debe ser:

*   **Prioridad de habilidad:** Debes subir **"La Espada de los Darkin" (Q)**.
    *   **Razonamiento:** Es tu fuente principal de daño y control de masas. Al nivel 1, te permite realizar el *poke* o el intercambio de daño manteniéndote a una distancia segura. Si logras conectar el "borde" (el *sweet spot*) de la Q, no solo infliges daño aumentado, sino que también aplicas un efecto de levantamiento (*knock-up*), lo que te permite ganar el intercambio sin recibir daño de retorno significativo de la mayoría de los campeones cuerpo a cuerpo.

*   **Power Spike en los primeros niveles:** Tu mayor pico de poder reside en el **Nivel 3**.
    *   **Razonamiento:** En el nivel 3, desbloqueas el combo completo de tus habilidades básicas: **Q, W y E**.
        *   La **Q** te proporciona el daño y control.
        *   La **W (Cadenas infernales)** te permite asegurar el impacto de las siguientes activaciones de tu Q al ralentizar al enemigo y atraerlo al centro.
        *   La **E (Deslizamiento oscuro)** es crítica aquí: te permite reposicionarte para asegurar el *sweet spot* de la Q mientras infliges daño, y te otorga **omnivampirismo**, permitiéndote sostener el intercambio de vida de manera mucho más eficiente que tu oponente.

**Consejo táctico:** No busques un intercambio prolongado en el nivel 1. Utiliza la Q para desgastar, y espera al nivel 3 para buscar una rotación completa de habilidades que fuerce un *flash* o asegure una ventaja de vida definitiva en la línea.


[🔧 ORIGEN DE LA RESPUESTA: Base de Datos Local (RAG - JSON)]
------------------------------------------------------------

🔹 Invocador: El rival se ha comprado Corazón de Acero de primer ítem. ¿Qué objeto debo comprar yo para contrarrestar su escalado de vida y no perder el 1v1 en línea?

🔸 Asistente Experto:


Para contrarrestar un **Corazón de Acero** (Heartsteel) en fase de líneas, el objetivo principal no es solo comprar daño, sino **daño porcentual de vida** y **penetración de armadura**.

Basándome en la premisa de que estás jugando con un coloso (como Aatrox, según tu consulta anterior, u otros campeones similares), aquí tienes la mejor estrategia de itemización:

*   **Tu objeto clave: Desgarrador Divino (o Eclipse, dependiendo del caso):**
    *   Si buscas una respuesta directa al escalado de vida, el **Desgarrador Divino** (si está disponible en tu versión actual) o el **Eclipse** son fundamentales. El **Eclipse** te otorga daño basado en la vida máxima del objetivo, lo cual neutraliza la ventaja que el rival obtiene al acumular salud con el Corazón de Acero.

*   **Opción de daño por vida máxima: Rencor de Serylda o Recordatorio Mortal:**
    *   Al comprar el Corazón de Acero, el rival inevitablemente también ganará una cantidad significativa de armadura en su *build* posterior. Priorizar un objeto con **penetración de armadura porcentual** (como el **Rencor de Serylda**) te permitirá seguir perforando su defensa aunque tenga una barra de vida masiva.

*   **El factor "Daño por Vida Máxima":**
    *   Si tu campeón permite el uso de **Botarruinas** (si es un atacante básico) o **Desgarrador**, priorízalos. El daño porcentual es el único lenguaje que entiende un rival que apuesta todo a la vida máxima.

**Análisis Estratégico para el 1v1:**
1.  **No busques el *all-in* prolongado:** El Corazón de Acero se carga cuando el rival te golpea. Si peleas de forma extendida, le permites aplicar la pasiva del objeto varias veces.
2.  **Gestiona los tiempos:** Si ves que el rival tiene la carga lista (el sonido característico del objeto), **retrocede**. Es mejor ceder unos súbditos que permitirle obtener 50-100 de vida máxima gratis.
3.  **Prioriza el escalado:** Si compras objetos con daño porcentual, escalarás mejor que él en el *mid-game*, ya que el Corazón de Acero es un objeto de "bola de nieve" que se vuelve menos efectivo a medida que tus objetos de penetración comienzan a completar su *build*.

**Nota:** Si el rival está usando campeones como **Skarner, Mordekaiser o Nasus** (mencionados en tu contexto), recuerda que el daño porcentual es obligatorio, ya que estos campeones escalan sus propias defensas o curaciones basándose en los intercambios largos que el Corazón de Acero fomenta.


[🔧 ORIGEN DE LA RESPUESTA: Base de Datos Local (RAG - JSON)]
------------------------------------------------------------

🔹 Invocador: Estoy en una situación de macrogame complicada: hemos ganado una pelea en top, pero el dragón está activo y el equipo enemigo está pusheando mid. ¿Qué es mejor: priorizar la torre de tier 1 en top para abrir el mapa o rotar inmediatamente a asegurar el dragón?

🔸 Asistente Experto:


...


[🔧 ORIGEN DE LA RESPUESTA: Base de Datos Local (RAG - JSON)]
------------------------------------------------------------

🔹 Invocador: Mi equipo es full AP y nos falta daño físico para cerrar la partida. De todos los campeones que tienes en tu base de datos, ¿cuál encaja mejor para balancear la composición y ser efectivo en teamfights de late game?

🔸 Asistente Experto:


Analizando los campeones disponibles en tu base de datos (**Aphelios, Alistar, Dr. Mundo**) y considerando el problema de un equipo **full AP**, la elección estratégica es clara:

### El Campeón Recomendado: Aphelios

Para balancear una composición que sufre por la falta de daño físico (*AD*), **Aphelios** es la opción superior. Aquí el razonamiento estratégico:

*   **Daño Físico Puro y Escalado de *Late Game*:** A diferencia de los otros candidatos, Aphelios es un tirador (*ADC*) que escala extremadamente bien con objetos de crítico y penetración. En una composición full AP, el equipo enemigo probablemente priorizará ítems de **Resistencia Mágica** (como *Fuerza de la Naturaleza* o *Máscara Abisal*). Aphelios obligará al rival a dividir sus recursos defensivos entre Armadura y Resistencia Mágica, anulando la eficiencia de su *itemización*.
*   **Versatilidad de Kit (Sinergia de Armas):**
    *   **Calibrum (Rifle):** Te permite castigar a los enemigos desde una distancia segura mientras tus aliados AP inician.
    *   **Severum (Pistola):** Te da *sustain* en peleas prolongadas.
    *   **Gravitum (Cañón):** Ofrece el control de masas (*CC*) que un equipo AP suele carecer, facilitando que tus aliados lancen sus habilidades de área (aoe) sobre objetivos inmovilizados.
*   **Eficiencia en Teamfights:** Un equipo full AP suele ser "de ráfaga" (*burst*). Aphelios aporta el **daño sostenido (DPS)** necesario para destruir tanques enemigos que hayan acumulado mucha vida, especialmente si usas **Infernum** (lanzallamas) para aplicar daño en área durante las peleas de equipo.

### ¿Por qué descartar las otras opciones?
*   **Alistar:** Aunque proporciona un gran *engage* y *peel* (protección), no soluciona tu carencia de daño físico. Añadir un Alistar a un equipo full AP solo hace que el equipo sea más resistente, pero incapaz de derribar torres o atravesar una composición con alta regeneración de vida o escudos.
*   **Dr. Mundo:** Si bien es un coloso poderoso, su daño es mayoritariamente mágico (debido a su escalado con vida y la naturaleza de sus habilidades en el contexto actual). Añadir a Mundo no corrige el déficit de daño físico de tu composición; simplemente añadirías más *AP/Magic Damage* a la mezcla.

### Recomendación Táctica para el cierre de partida:
Si seleccionas a **Aphelios**, tu equipo debe jugar en torno a ti como el "seguro de vida". Dado que el enemigo se armará contra AP, tu **penetración de armadura** (ítems como *Recordatorio Mortal* o *Serylda*) será el factor determinante que les impedirá ganar las peleas de equipo en el *late game*.

**Posicionamiento:** Mantente en la retaguardia. Tu equipo AP debe gastar sus *cooldowns* para obligar al enemigo a gastar sus defensas, y tú debes entrar una vez que las amenazas de *backline* enemigas hayan sido comprometidas por tus aliados.


[🔧 ORIGEN DE LA RESPUESTA: Base de Datos Local (RAG - JSON)]
------------------------------------------------------------

🔹 Invocador: El equipo enemigo tiene un Zed fedeado y mucha capacidad de CC. ¿Debería cambiar mi build estándar para priorizar supervivencia o es mejor ajustar mi estilo de juego para no ser el objetivo principal en las teamfights?

🔸 Asistente Experto:


Contra un **Zed** fedeado, la supervivencia no es una opción, es una **necesidad estratégica**. Si intentas mantener tu *build* estándar, Zed te eliminará en su rotación de habilidades (*burst*) antes de que puedas aportar tu daño al equipo.

Aquí tienes el análisis para ajustar tu configuración y tu juego:

### 1. Ajuste de Build (Prioriza la Supervivencia Activa)
No sacrifiques todo tu daño, pero integra elementos de **supervivencia defensiva** que anulen el daño explosivo de Zed:

*   **Reloj de Arena de Zhonya (Si usas AP) o Ángel Guardián (Si usas AD):** 
    *   **Zhonya** es el *hard counter* definitivo contra la definitiva de Zed (**Marca de la Muerte**). Debes activarlo justo antes de que la marca explote para ignorar el daño acumulado.
    *   **Ángel Guardián** te da una segunda oportunidad. Si Zed gasta todos sus recursos en matarte, su equipo quedará 4v5 mientras tú revives.
*   **Encantamiento o Ítems de reducción de daño:** Si el enemigo tiene mucho **CC**, considera el **Crisol de Mikael** (si eres soporte) o simplemente asegúrate de tener **Botas de Mercurio**. La tenacidad es vital; si Zed te lanza su combo y logras sobrevivir con poca vida, el CC enemigo será lo que te termine de matar. Necesitas reducir ese tiempo de inmovilización para poder reposicionarte.
*   **Faja de Mercurio (QSS):** Es obligatoria si el CC enemigo es de tipo supresión o *stun* encadenado. Si Zed te marca, y al mismo tiempo un CC enemigo te inmoviliza, el QSS es tu única vía de escape.

### 2. Ajuste de Estilo de Juego (El factor "Target Selection")
Cambiar tu *build* es solo el 50% de la solución. El otro 50% es tu disciplina táctica:

*   **El "Fog of War" es tu mayor enemigo:** Un Zed fedeado no está en la pelea, está **esperando en la sombra**. Nunca te muestres en la línea si no ves a Zed en el mapa. Si no aparece, asume que está flanqueando tu posición.
*   **Posicionamiento de "Segunda Línea":** No seas el primero en entrar. Deja que tu *tank* (ej. Alistar o un campeón con CC) inicie. Zed buscará al objetivo más frágil y vulnerable. Si te mantienes cerca de tus aliados que tienen CC de área, puedes forzar a Zed a cometer un error si intenta entrar agresivamente.
*   **Control de Visión:** Compra un **Centinela de Control** y colócalo en los flancos, no solo en los arbustos de la línea. Zed necesita ángulos de entrada. Si le quitas la capacidad de sorprenderte, su efectividad cae un 60%.

### Razonamiento Estratégico:
*   **¿Cambiar la build o el estilo?** **Ambos.** Si solo cambias la *build*, te faltará daño para ganar la pelea. Si solo cambias el estilo, Zed seguirá encontrando oportunidades de *pick-off* porque eres un objetivo prioritario.
*   **Mentalidad de equipo:** Comunícate con tu equipo. Si tú eres el objetivo principal, diles: *"Guarden el CC para cuando Zed aparezca"*. Un Zed fedeado es un campeón de "todo o nada". Si fallan su combo o son interrumpidos por un CC justo después de entrar, se quedan totalmente expuestos y mueren al instante.

**Conclusión:** Prioriza un objeto defensivo que ofrezca **invulnerabilidad temporal o revivificación**. En el *macrogame*, obliga a Zed a ser el que tenga que arriesgarse; no le regales la iniciación caminando solo por la jungla.


[🔧 ORIGEN DE LA RESPUESTA: Base de Datos Local (RAG - JSON)]
------------------------------------------------------------

Canal cerrado. ¡Buena suerte en la Grieta, Invocador!🧙‍♂️
